# Replication - Pairs Trading with a Novel Graphical Matching Approach

Ce notebook initialise la replication avec une pipeline de donnees rigoureuse:
- univers historique S&P 500 (point-in-time),
- prix journaliers ajustes (dividendes + splits),
- benchmark S&P 500,
- mapping statique instrument (ISIN, BBGID, nom, devise, secteur).

## 1) Checklist des donnees a telecharger

Minimum strict pour reproduire le papier:
1. Constituants historiques S&P 500 (monthly point-in-time)
2. Daily adjusted close pour tous les titres
3. Daily benchmark S&P 500
4. Mapping statique ticker -> ISIN/BBGID/name/currency/sector

Dictionnaire de travail du notebook:
- `prices`, `benchmark`, `universe`, `static_metadata` sont les datasets core
- `log_prices`, `daily_returns`, `daily_log_returns` sont les features de base pour regression/ADF

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)

# Trouve la racine du projet meme si le notebook est lance depuis un sous-dossier
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise RuntimeError('Impossible de trouver le dossier src depuis le repertoire courant.')

sys.path.insert(0, str(ROOT / 'src'))

from bloomberg import BloombergDataProvider, DEFAULT_STATIC_FIELDS  # type: ignore

print(f'Project root: {ROOT}')

In [ ]:
# Parametres replication papier
INDEX_NAME = 'SPX'
BENCHMARK_TICKER = 'SPX Index'  # ou 'SPXT Index' pour total return
BACKTEST_START = '2017-01-01'
BACKTEST_END = '2023-05-31'
LOOKBACK_YEARS = 2

# Fenetre de donnees brute: commence 2 ans avant le debut du backtest
DATA_START = (pd.Timestamp(BACKTEST_START) - pd.DateOffset(years=LOOKBACK_YEARS)).strftime('%Y-%m-%d')
DATA_END = BACKTEST_END

# Reutilise les parquets copies depuis inspiration
RAW_DIR = ROOT / 'src' / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REFRESH = False

print('Config:')
print(f'  Index         : {INDEX_NAME}')
print(f'  Benchmark     : {BENCHMARK_TICKER}')
print(f'  Data window   : {DATA_START} -> {DATA_END}')
print(f'  Cache path    : {RAW_DIR}')
print(f'  Force refresh : {FORCE_REFRESH}')

In [ ]:
provider = BloombergDataProvider(
    index_name=INDEX_NAME,
    start_date=DATA_START,
    end_date=DATA_END,
    benchmark_ticker=BENCHMARK_TICKER,
    output_path=RAW_DIR,
)

data = provider.fetch_or_load(force_refresh=FORCE_REFRESH)

prices = data['prices']
benchmark = data['benchmark']
static_metadata = data['static_metadata']
universe = data['universe']

# Optionnels (pas indispensables a la replication stricte, mais utiles)
pe_ratios = data['pe_ratios']
risk_free = data['risk_free']

In [ ]:
# Nettoyage + controles qualite
prices.index = pd.to_datetime(prices.index).sort_values()
benchmark.index = pd.to_datetime(benchmark.index).sort_values()
universe['date'] = pd.to_datetime(universe['date'])

assert not prices.empty, 'prices est vide'
assert not benchmark.empty, 'benchmark est vide'
assert not universe.empty, 'universe est vide'

summary = pd.DataFrame({
    'metric': [
        'n_assets_prices',
        'n_rows_prices',
        'prices_start',
        'prices_end',
        'benchmark_start',
        'benchmark_end',
        'n_universe_rows',
        'n_universe_unique_tickers',
        'static_metadata_rows',
    ],
    'value': [
        prices.shape[1],
        prices.shape[0],
        prices.index.min(),
        prices.index.max(),
        benchmark.index.min(),
        benchmark.index.max(),
        len(universe),
        universe['ticker'].nunique(),
        len(static_metadata),
    ]
})
summary

In [ ]:
# Features de base pour le papier
prices = prices.sort_index()
log_prices = np.log(prices.replace(0.0, np.nan))
daily_returns = prices.pct_change(fill_method=None)
daily_log_returns = log_prices.diff()

rebalance_dates = pd.date_range(
    start=pd.Timestamp(BACKTEST_START),
    end=pd.Timestamp(BACKTEST_END),
    freq='ME',
)

print(f'Rebalance dates: {len(rebalance_dates)}')
print(f'First 3: {list(rebalance_dates[:3])}')
print(f'Last 3 : {list(rebalance_dates[-3:])}')

In [ ]:
# Sauvegarde de jeux de donnees canoniques pour les prochaines etapes
log_prices.to_parquet(RAW_DIR / 'log_prices.parquet')
daily_returns.to_parquet(RAW_DIR / 'daily_returns.parquet')
daily_log_returns.to_parquet(RAW_DIR / 'daily_log_returns.parquet')
benchmark.to_frame('benchmark').to_parquet(RAW_DIR / 'benchmark.parquet')

universe_enriched = universe.merge(
    static_metadata[['ticker', *[c.lower() for c in DEFAULT_STATIC_FIELDS]]],
    on='ticker',
    how='left',
)
universe_enriched.to_parquet(RAW_DIR / 'universe_enriched.parquet')

pd.DataFrame({'rebalance_date': rebalance_dates}).to_parquet(
    RAW_DIR / 'rebalance_dates.parquet'
)

print('Datasets de base sauvegardes.')

## 2) Prochaine etape

Construire le moteur de selection des paires par date de rebalancement:
1. fenetre rolling 2 ans de `log_prices`,
2. regressions OLS pairwise,
3. tests ADF sur residus,
4. edge list `stock_i, stock_j, adf_t_stat, adf_p_value, weight=-adf_t_stat`,
5. selection baseline (top paires) vs maximum-weight matching.